### Goal: 
Group customers based on spending and activity to support campaign targeting.
### Why it matters: 
Enables personalized offers and engagement.
### How to do it:

### Use RFM logic based on order_items:
- Recency: Days since last purchase
- Frequency: Number of purchases in last N months
- Monetary: Total spend in last N months
### Segment:
- VIPs: High R, F, M
- New Customers: Low F, high R
- Churn Risk: Low R, low F

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')

In [0]:
snapshot_date = (df_fact_order.agg(F.max('CREATION_TIME_UTC').alias('snapshot_date')).first()['snapshot_date'])

### Filtering for last 12 Months (N=12)

In [0]:
df_fact_order_12mon = df_fact_order.filter(F.col('CREATION_TIME_UTC') >= F.add_months(F.lit(snapshot_date),-12))

### Userwise Aggregation of recency,frequency,Total_amt

In [0]:
df_user_metrics = (df_fact_order_12mon.groupBy('USER_ID')
              .agg(F.datediff(F.lit(snapshot_date),F.max('CREATION_TIME_UTC')).alias('recency'),
                   F.countDistinct('order_id').alias('frequency'),
                   F.round(F.sum('ITEM_PRICE'),2).alias('total_amt'))
    )
#df_user_metrics.orderBy(F.col('recency').desc()).display()

### Segment users based on the condition given
Assumed High means > 50% vice versa for low

In [0]:
recency_window = Window.orderBy(F.col('recency').desc())
freq_window = Window.orderBy(F.col('frequency').desc())
total_amt_window = Window.orderBy(F.col('total_amt').desc())

In [0]:
df_user_segment = (df_user_metrics.withColumn('rnk_recency',F.percent_rank().over(recency_window))
                    .withColumn('rnk_frq',F.percent_rank().over(freq_window))
                    .withColumn('rnk_amt',F.percent_rank().over(total_amt_window))
                    .withColumn('segment',F.when(
                        (F.col('rnk_recency') > 0.5) & 
                        (F.col('rnk_frq') > 0.5) & 
                        (F.col('rnk_amt') > 0.5),'VIPs')
                                .when(
                            (F.col('rnk_recency') > 0.5) & 
                        (F.col('rnk_frq') < 0.5),'New Customers')
                            .when(
                            (F.col('rnk_recency') < 0.5) & 
                        (F.col('rnk_frq') < 0.5),'Churn Risk')
                            .otherwise('NA'))
                    )
#df_user_segment.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.customer_segmentation

In [0]:
df_user_segment.write.mode('append').saveAsTable('global_partner_project.mart.customer_segmentation')